In [30]:
import json
from dotenv import load_dotenv

load_dotenv()

import importlib

import agents
from agents.templates.play_zero_agent import PlayZeroAgent

agents_module = importlib.reload(agents)

from agents.structs import FrameData, GameState

import textwrap

def print_wrapped_text(text: str, width: int = 80):
    """
    Print the given text with word-wrapped lines for better readability in the terminal.

    Args:
        text (str): The input text to be printed.
        width (int): The maximum line width before wrapping. Default is 80.
    """
    wrapper = textwrap.TextWrapper(width=width)
    paragraphs = text.strip().split("\n\n")

    for paragraph in paragraphs:
        wrapped = wrapper.fill(paragraph)
        print(wrapped + "\n")

#  Agent.__init__() missing 5 required positional arguments: 'card_id', 'game_id', 'agent_name', 'ROOT_URL', and 'record'
play_zero_agent: PlayZeroAgent = agents_module.templates.play_zero_agent.PlayZeroAgent(
    card_id="play_zero_agent",
    game_id="play_zero_agent",
    agent_name="PlayZeroAgent",
    ROOT_URL="http://localhost:8000",
    record=False,
)

video_path = "/workspaces/ARC-AGI-3-Agents/recordings/game_analysis_ls20-f340c8e5138e.mp4"
scorecard_file_path = "/workspaces/ARC-AGI-3-Agents/recordings/ls20-f340c8e5138e.playzeroagent.gemini-2.5-flash.with-observe.gemini-2.5-flash.8aefc1ea-8e65-41ec-9272-a8faff24ddb1.recording.jsonl"

with open(scorecard_file_path, "r") as file:
    grid_jsons = [json.loads(line) for line in file]
frames = [FrameData(**frame_json["data"]) for frame_json in grid_jsons]
frame_start = frames[0]
frame_2 = frames[1]
frame_3 = frames[2]
frame_end = frames[-3]
current_frame = frame_end

In [32]:
logical_analysis_actions_summary = play_zero_agent.generate_logical_analysis_summary(frames[:55])
print(f"Logical Analysis Actions Summary:\n\n {logical_analysis_actions_summary}")

Logical Analysis Actions Summary:

 Out of all user inputs recorded during gameplay:

- **W** was used 15 times, with 4 inputs having no effect on gameplay.
- **A** was used 7 times, and all had an effect on the game.
- **S** was used 5 times, with 2 inputs having no effect on gameplay.
- **D** was used 10 times, with 3 inputs having no effect on gameplay.
- **CLICK** was used 8 times, and all had no effect on the game.


In [ ]:
# Eval prompt for multiple_hypothesis_text

EVAL_PROMPT = """Give score and reason of whether the multiple hypothesis can be used to generate the expected goal

Expected Goal: <expected_goal>{expected_goal}</expected_goal>

Multiple Hypothesis Text: <multiple_hypothesis_text>{multiple_hypothesis_text}</multiple_hypothesis_text>

Example output json:
```json
{{
    "reason": "<max of 50 words>",
    "score": "<score from 0 to 10>"
}}
```
"""

def evaluate_multiple_hypothesis_text(multiple_hypothesis_text: str, expected_goal: str):
    prompt = EVAL_PROMPT.format(
        expected_goal=expected_goal,
        multiple_hypothesis_text=multiple_hypothesis_text
    )
    response = play_zero_agent.client.chat.completions.create(
        model="gemini-2.5-flash",
        messages = [
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    json_text = response.choices[0].message.content.strip()
    json_text = play_zero_agent.extract_first_json_block(json_text)
    json_data = json.loads(json_text)
    return json_data


In [ ]:
multiple_hypothesis_text = play_zero_agent.generate_multiple_random_hypothesis_from_video(
    video_path,
    logical_analysis_actions_summary=logical_analysis_actions_summary,
)
expected_goal_ls20_1 = "You need to move the \"Orange-Capped Blue Block (6x7)\" to the target \"8x7Grid_BlackHead_BlueEye_WhiteSnout\""
evaluation_result = evaluate_multiple_hypothesis_text(
    multiple_hypothesis_text=multiple_hypothesis_text,
    expected_goal=expected_goal_ls20_1
)

print_wrapped_text(f"Evaluation Result:\n\n {evaluation_result}")

In [34]:
goal = play_zero_agent.generate_top_hypothesis(
    multiple_hypothesis_text=multiple_hypothesis_text,
    logical_analysis_actions_summary=logical_analysis_actions_summary,
)
print_wrapped_text(f"Goal:\n\n {goal}"
)

Goal:

 The winning Multi-Stage Delivery hypothesis with significant win impact on the
game is as follows:

The game's progression and winning condition revolve around a multi-stage
puzzle-solving process. The primary objective is to strategically maneuver the
**Orange_Blue_Combined_Block** (a 1x3 pixel unit, formed by the
**Orange_Rectangle_Block** (1x2 pixels, orange) and the **Blue_Square_Block**
(1x1 pixel, blue)) through the **Grey_Maze_Area** (large irregular dark grey
block). This is achieved by using the **White_Player_Block_with_Blue_Dot**
(approximately 1x1 grid unit with a small blue pixel) to push the
**Orange_Rectangle_Block**. The immediate sub-goal for each puzzle, focusing on
elements **Orange_Blue_Combined_Block**, **White_Player_Block_with_Blue_Dot**,
**Blue_Dot_Target**, and **Black_Square_Target_Area**, is to precisely place the
**Blue_Square_Block** component onto the **Blue_Dot_Target** (small, single blue
pixel) located within the fixed **Black_Square_Target_Area

In [ ]:
# multiple_hypothesis_text = frame_end.action_input.reasoning["multiple_hypothesis_text"]
# logical_analysis_actions_summary = frame_end.action_input.reasoning["logical_analysis_actions_summary"]
goal = play_zero_agent.generate_top_hypothesis(
    multiple_hypothesis_text=multiple_hypothesis_text,
    logical_analysis_actions_summary=logical_analysis_actions_summary,
)

In [12]:
# print_wrapped_text(f"Multiple Hypothesis Text:\n\n {multiple_hypothesis_text}")
print_wrapped_text(f"Logical Analysis Actions Summary:\n\n {logical_analysis_actions_summary}")
# print_wrapped_text(f"Goal:\n\n {goal}")

Logical Analysis Actions Summary:

 Out of all user inputs recorded during gameplay:

- **W** was used 15 times, with 4 inputs having no effect on gameplay. - **A**
was used 7 times, and all had an effect on the game. - **S** was used 5 times,
with 2 inputs having no effect on gameplay. - **D** was used 10 times, with 3
inputs having no effect on gameplay. - **CLICK** was used 8 times, and all had
no effect on the game.

